# Predicción de IMPACTO_FRAUDE (ordinal 0-3)

Clasificación multiclase del impacto del fraude.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score)
import xgboost as xgb
import lightgbm as lgb
try:
    import catboost as cb
    CATBOOST_AVAIL = True
except ImportError:
    CATBOOST_AVAIL = False
    print('CatBoost no instalado — se omite')
sys.path.append(str(Path.cwd().parent))
from model.feature_engineering import *
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('OK')

## 1. Carga

In [ ]:
DATA_PATH = Path.cwd().parent / 'Notebooks' / 'data' / 'dataset_fraude.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nDistribución IMPACTO_FRAUDE:')
print(df['IMPACTO_FRAUDE'].value_counts().sort_index())
df.head(3)

In [ ]:
# Eliminar el otro target para evitar data leakage
df.drop(columns=['IS_FRAUD'], inplace=True)
print('IS_FRAUD eliminado — shape:', df.shape)

## 2. Train/Test split

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['IMPACTO_FRAUDE']
)
print(f'Train: {train_df.shape[0]}  Test: {test_df.shape[0]}')

## 3. Feature Engineering

In [ ]:
fe = FeatureEngineer(encode_target='IMPACTO_FRAUDE', random_state=42)
X_train = fe.fit_transform(train_df)
y_train = X_train.pop('IMPACTO_FRAUDE').values
X_test = fe.transform(test_df)
y_test = X_test.pop('IMPACTO_FRAUDE').values
num_feats = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
Xtr = X_train.copy(); Xte = X_test.copy()
Xtr[num_feats] = scaler.fit_transform(X_train[num_feats])
Xte[num_feats] = scaler.transform(X_test[num_feats])
print(f'Features: {Xtr.shape[1]}')

## 4. Modelos (multiclase)

In [ ]:
models = [
    ('LogisticRegression', LogisticRegression(solver='lbfgs', max_iter=1000),
     {'C': [0.1, 1, 10]}),
    ('RandomForest', RandomForestClassifier(random_state=42),
     {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
    ('GradientBoosting', GradientBoostingClassifier(random_state=42),
     {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('XGBoost', xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='mlogloss'),
     {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('LightGBM', lgb.LGBMClassifier(random_state=42, verbose=-1),
     {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}),
    ('ExtraTrees', ExtraTreesClassifier(random_state=42),
     {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}),
]
if CATBOOST_AVAIL:
    models.append(('CatBoost', cb.CatBoostClassifier(random_state=42, verbose=False, loss_function='MultiClass'),
                   {'iterations': [100, 200], 'depth': [4, 6], 'learning_rate': [0.05, 0.1]}))

results = []; bests = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, est, params in models:
    print(f'\n>>> {name}')
    gs = GridSearchCV(est, params, cv=cv, scoring='f1_weighted', n_jobs=-1)
    gs.fit(Xtr, y_train)
    bests[name] = gs.best_estimator_
    yp = gs.predict(Xte)
    results.append({'modelo': name, 'cv_f1': gs.best_score_,
                    'test_f1': f1_score(y_test, yp, average='weighted')})
    print(f'  CV={gs.best_score_:.4f}  Test F1={results[-1]["test_f1"]:.4f}')

res = pd.DataFrame(results).sort_values('test_f1', ascending=False)
print('\n' + '='*50)
print(res.round(4).to_string(index=False))

## 5. Mejor modelo

In [ ]:
best_name = res.iloc[0]['modelo']
best = bests[best_name]
print(f'Mejor: {best_name}')
print(classification_report(y_test, best.predict(Xte), digits=4))

In [ ]:
cm = confusion_matrix(y_test, best.predict(Xte))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}'); plt.tight_layout(); plt.show()

In [ ]:
if hasattr(best, 'feature_importances_'):
    imp = pd.DataFrame({'f': Xtr.columns, 'imp': best.feature_importances_}).sort_values('imp', ascending=False)
    sns.barplot(data=imp.head(20), x='imp', y='f', palette='viridis')
    plt.title(f'Top-20 - {best_name}'); plt.tight_layout(); plt.show()

## 6. Conclusiones

In [ ]:
print('=== FINAL ===')
print(f'Mejor: {best_name}  F1={res.iloc[0]["test_f1"]:.4f}')
print(df['IMPACTO_FRAUDE'].value_counts().sort_index().to_string())